<a href="https://colab.research.google.com/github/Sarahnjunge/Enveda_Casmi_2026/blob/main/notebookb48a93fa8e.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

enveda_casmi26_molecule_id_mass_spectra_path = kagglehub.competition_download('enveda-CASMI26-molecule-id-mass-spectra')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Enveda CASMI 2026 - baseline: mass-filtered spectral library search
# Designed to run as a Kaggle notebook (paste cells split by "# %%") or as a script.
#
# Idea
#   1. Turn every spectrum into a sparse vector: [fragment-m/z bins | neutral-loss bins].
#      Cosine on this vector ~ a cheap "modified cosine" (loss block lets analogs match).
#   2. For each test molecule, take its neutral mass (from precursor + adduct), pull all
#      reference spectra within a growing mass window, and score each reference structure
#      = mean over the molecule's spectra of (best cosine over that structure's spectra).
#   3. Rank by (mass tier, score) and emit the top 25 SMILES.
#
# Env vars: DATA_DIR (folder with train/test/sample_submission), OUT_DIR, QUICK=1 (subsample train)

# %% imports / config
import os, re, glob, time, sys
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
import scipy.sparse as sp

def _find_dir():
    if os.environ.get("DATA_DIR"):
        return os.environ["DATA_DIR"]
    hits = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
    return os.path.dirname(hits[0]) if hits else "."

DATA_DIR = _find_dir()
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
QUICK = int(os.environ.get("QUICK", "0"))

BIN = 0.02            # m/z bin width (Da)
MAX_MZ = 1300.0
DIM = int(MAX_MZ / BIN) + 4
TOPN = 64             # peaks kept per spectrum
MIN_REL = 0.01        # relative intensity floor
TOPK = 25
TIERS_DA = None       # filled per molecule (first tier is ppm based)
PPM = 15.0
TIER_ABS = [0.05, 0.5, 3.0, 15.0]   # widening windows (Da) used to fill up to 25 candidates
MAX_CAND_SPECTRA = 150_000
MAX_PPM_ERR_TRAIN = 20.0

PROTON, ELECTRON = 1.007276, 0.000549
H2O, NH4, NA, K, CL, FA = 18.010565, 18.033823, 22.989221, 38.963158, 34.969402, 46.005479
ADDUCT_DELTA = {   # ion m/z = M + delta
    "[M+H]+": PROTON, "[M+NH4]+": NH4, "[M-H2O+H]+": PROTON - H2O,
    "[M-2H2O+H]+": PROTON - 2 * H2O, "[M+Na]+": NA, "[M+K]+": K,
    "[M-H]-": -PROTON, "[M-H2O-H]-": -PROTON - H2O,
    "[M+CH2O2-H]-": FA - PROTON, "[M+Cl]-": CL,
}
EL = {"H": 1.00782503, "C": 12.0, "N": 14.0030740, "O": 15.9949146, "S": 31.9720707,
      "P": 30.9737620, "F": 18.9984032, "Cl": 34.9688527, "Br": 78.9183376, "I": 126.904473,
      "Na": 22.9897693, "K": 38.9637069, "Si": 27.9769265, "B": 11.0093054, "Se": 79.9165218,
      "Fe": 55.9349375, "Mg": 23.9850417, "Ca": 39.9625912, "Zn": 63.9291422, "Cu": 62.9295975,
      "As": 74.9215965, "Li": 7.01600455, "Al": 26.9815386, "Mn": 54.9380451, "Co": 58.9331950,
      "Ni": 57.9353429, "Sn": 119.9022016, "Pt": 194.9647917, "Hg": 201.970643, "Sb": 120.9038157}

def formula_mass(f):
    if not isinstance(f, str) or not f:
        return np.nan
    tot = 0.0
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", f):
        if el not in EL:
            return np.nan
        tot += EL[el] * (int(n) if n else 1)
    return tot

# %% spectrum -> sparse vector
def spectra_to_csr(mz_col, int_col, prec):
    """mz_col / int_col: pyarrow ListArray; prec: float array. Fully vectorised."""
    n = len(prec)
    off = mz_col.offsets.to_numpy()
    off = off - off[0]
    lens = np.diff(off)
    mz = mz_col.flatten().to_numpy(zero_copy_only=False).astype(np.float64)
    it = int_col.flatten().to_numpy(zero_copy_only=False).astype(np.float64)
    sid = np.repeat(np.arange(n), lens)
    keep = (mz < prec[sid] - 0.5) & (mz >= 30.0) & (it >= MIN_REL)
    sid, mz, it = sid[keep], mz[keep], it[keep]
    order = np.lexsort((-it, sid))                # by spectrum, then intensity desc
    sid, mz, it = sid[order], mz[order], it[order]
    first = np.searchsorted(sid, np.arange(n))
    rank = np.arange(len(sid)) - first[sid]
    k = rank < TOPN
    sid, mz, it = sid[k], mz[k], it[k]
    norm = np.sqrt(np.bincount(sid, weights=it, minlength=n))   # = ||sqrt(I)||
    val = np.sqrt(it) / norm[sid] * np.sqrt(0.5)                # each block unit-norm * sqrt(.5)
    fb = np.rint(mz / BIN).astype(np.int64)
    lb = np.rint((prec[sid] - mz) / BIN).astype(np.int64)
    ok = (fb < DIM) & (lb < DIM) & (lb >= 0)
    sid, fb, lb, val = sid[ok], fb[ok], lb[ok], val[ok]
    rows = np.concatenate([sid, sid])
    cols = np.concatenate([fb, lb + DIM])
    data = np.concatenate([val, val]).astype(np.float32)
    return sp.csr_matrix((data, (rows, cols)), shape=(n, 2 * DIM), dtype=np.float32)

def read_spectra(path, meta_cols, batch=200_000, max_rows=None):
    pf = pq.ParquetFile(path)
    spec_cols = ["ms2_mzs", "ms2_normalized_intensities"]
    metas, mats, seen = [], [], 0
    for b in pf.iter_batches(batch_size=batch, columns=meta_cols + spec_cols):
        t = pa.Table.from_batches([b])
        meta = t.select(meta_cols).to_pandas()
        prec = meta["precursor_mz"].to_numpy(float)
        mats.append(spectra_to_csr(b.column("ms2_mzs"), b.column("ms2_normalized_intensities"), prec))
        metas.append(meta)
        seen += len(meta)
        print(f"  read {seen:,} spectra", flush=True)
        if max_rows and seen >= max_rows:
            break
    return pd.concat(metas, ignore_index=True), sp.vstack(mats, format="csr")

# %% build / load reference
TRAIN_META = ["inchikey14", "normalized_smiles", "molecular_formula", "ionization_mode",
              "precursor_mz", "precursor_error_ppm", "ingest_lib", "adduct"]
TEST_META = ["molecule_id", "spectrum_id", "adduct", "ionization_mode", "precursor_mz",
             "base_peak_intensity"]

def build_reference():
    cache_meta = os.path.join(OUT_DIR, "ref_meta.parquet")
    cache_mat = os.path.join(OUT_DIR, "ref_mat.npz")
    if os.path.exists(cache_meta) and os.path.exists(cache_mat) and not QUICK:
        print("loading cached reference")
        return pd.read_parquet(cache_meta), sp.load_npz(cache_mat).tocsr()
    t0 = time.time()
    meta, mat = read_spectra(os.path.join(DATA_DIR, "train.parquet"), TRAIN_META,
                             max_rows=300_000 if QUICK else None)
    smass = meta["molecular_formula"].map(
        {f: formula_mass(f) for f in meta["molecular_formula"].dropna().unique()})
    good = (smass.notna() & meta["inchikey14"].notna() & meta["normalized_smiles"].notna()
            & meta["ionization_mode"].isin(["positive", "negative"])
            & ~(meta["precursor_error_ppm"].abs() > MAX_PPM_ERR_TRAIN)
            & (np.diff(mat.indptr) > 2)).to_numpy()
    meta = meta[good].reset_index(drop=True)
    meta["struct_mass"] = smass[good].to_numpy()
    mat = mat[good]
    meta.to_parquet(cache_meta)
    if not QUICK:
        sp.save_npz(cache_mat, mat)
    print(f"reference: {len(meta):,} spectra, {meta.inchikey14.nunique():,} structures, "
          f"{time.time()-t0:.0f}s")
    return meta, mat

def make_ref(meta, mat):
    sid, uniq = pd.factorize(meta["inchikey14"])
    first = pd.Series(np.arange(len(meta))).groupby(sid).first().to_numpy()
    R = dict(
        mat=mat, sid=sid.astype(np.int32),
        smiles=meta["normalized_smiles"].to_numpy()[first],
        smass=meta["struct_mass"].to_numpy()[first],
        ik=np.asarray(uniq),
        mode=(meta["ionization_mode"] == "positive").to_numpy(),
        lib=meta["ingest_lib"].to_numpy(),
    )
    R["mass"] = R["smass"][R["sid"]]
    R["order"] = np.argsort(R["mass"], kind="stable")
    R["mass_sorted"] = R["mass"][R["order"]]
    return R

# %% search
SPREAD = None
def _spread():
    global SPREAD
    if SPREAD is None:
        SPREAD = sp.diags([0.5, 1.0, 0.5], [-1, 0, 1], shape=(2 * DIM, 2 * DIM), format="csr",
                          dtype=np.float32)
    return SPREAD

def neutral_mass(meta):
    d = meta["adduct"].map(ADDUCT_DELTA)
    return (meta["precursor_mz"] - d).to_numpy(float)

def search_molecule(T, tmode, M, R, ref_ok):
    tols = np.array([PPM * 1e-6 * M] + TIER_ABS)
    idx = None
    for tol in tols:
        lo = np.searchsorted(R["mass_sorted"], M - tol, "left")
        hi = np.searchsorted(R["mass_sorted"], M + tol, "right")
        idx = R["order"][lo:hi]
        idx = idx[ref_ok[idx]]
        if len(np.unique(R["sid"][idx])) >= TOPK:
            break
    if len(idx) == 0:
        return []
    if len(idx) > MAX_CAND_SPECTRA:
        idx = idx[np.argsort(np.abs(R["mass"][idx] - M))[:MAX_CAND_SPECTRA]]
    S = (T @ R["mat"][idx].T).toarray()
    S *= (tmode[:, None] == R["mode"][idx][None, :])          # polarity must match
    sc = R["sid"][idx]
    o = np.argsort(sc, kind="stable")
    S, sc = S[:, o], sc[o]
    starts = np.r_[0, np.flatnonzero(np.diff(sc)) + 1]
    score = np.maximum.reduceat(S, starts, axis=1).mean(0)
    usid = sc[starts]
    dm = np.abs(R["smass"][usid] - M)
    tier = (dm[:, None] > tols[None, :]).sum(1)
    top = usid[np.lexsort((-score, tier))[:TOPK]]
    return list(top)

def run_search(Tmeta, T, R, ref_ok):
    """Tmeta needs molecule_id, ionization_mode, neutral_mass. T: csr rows aligned to Tmeta."""
    Ts = (T @ _spread()).tocsr()
    tmode_all = (Tmeta["ionization_mode"] == "positive").to_numpy()
    nm = Tmeta["neutral_mass"].to_numpy(float)
    out = {}
    groups = Tmeta.groupby("molecule_id").indices
    for i, (mol, rows) in enumerate(groups.items()):
        M = np.nanmedian(nm[rows])
        out[mol] = run = search_molecule(Ts[rows], tmode_all[rows], M, R, ref_ok) if np.isfinite(M) else []
        if i % 50 == 0:
            print(f"  searched {i}/{len(groups)} molecules", flush=True)
    return out

# %% validation on enveda-np-examples (same instrument/pipeline as test)
def validate(meta, mat, R):
    is_np = (R["lib"] == "enveda-np-examples")
    if is_np.sum() == 0:
        print("no enveda-np-examples rows found; skipping validation"); return
    rows = np.flatnonzero(is_np)
    Tmeta = pd.DataFrame({
        "molecule_id": R["ik"][R["sid"][rows]],
        "ionization_mode": np.where(R["mode"][rows], "positive", "negative"),
        "neutral_mass": np.where(
            meta["adduct"].iloc[rows].isin(ADDUCT_DELTA).to_numpy(),
            neutral_mass(meta.iloc[rows]), R["mass"][rows]),
    })
    T = mat[rows]
    np_structs = np.unique(R["sid"][rows])
    for name, ref_ok in [
        ("class-1-like (np-examples spectra removed, other libs kept)", ~is_np),
        ("hard (structure fully removed from reference)", ~np.isin(R["sid"], np_structs)),
    ]:
        res = run_search(Tmeta, T, R, ref_ok)
        hits = {1: 0, 5: 0, 25: 0}
        for mol, top in res.items():
            iks = [R["ik"][s] for s in top]
            for k in hits:
                hits[k] += mol in iks[:k]
        n = len(res)
        print(f"[{name}] molecules={n}  " + "  ".join(f"top{k}={v/n:.3f}" for k, v in hits.items()))

# %% main
def main():
    meta, mat = build_reference()
    R = make_ref(meta, mat)
    if os.environ.get("VALIDATE", "1") == "1":
        validate(meta, mat, R)
    print("loading test")
    tm, T = read_spectra(os.path.join(DATA_DIR, "test.parquet"), TEST_META)
    unk = set(tm["adduct"]) - set(ADDUCT_DELTA)
    if unk:
        print("WARNING unknown adducts:", unk)
    tm["neutral_mass"] = neutral_mass(tm)
    res = run_search(tm, T, R, np.ones(len(R["mass"]), bool))
    sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
    smiles = []
    for mol in sub["molecule_id"]:
        s = [R["smiles"][i] for i in res.get(mol, [])]
        s += ["CCO"] * (TOPK - len(s))
        smiles.append(";".join(s[:TOPK]))
    sub["smiles"] = smiles
    path = os.path.join(OUT_DIR, "submission.csv")
    sub.to_csv(path, index=False)
    print("wrote", path)

if __name__ == "__main__":
    main()